# 2D FWI

这个示例使用二阶 FDTD，并对介电常数和电导率采用分阶段反演。数据项按观测振幅无量纲化，TV 作为小权重的普通目标函数项；DeepGPR 返回的原始梯度不再做 RMS 或最大值重标定。


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from skimage import filters

# 只允许加载当前 DeepGPR 项目中的源码，不使用 site-packages 里的旧版本。
cwd = Path.cwd().resolve()
candidate_roots = (cwd, cwd.parent, cwd / "DeepGPR")
project_root = next(
    (path for path in candidate_roots if (path / "src" / "DeepGPR" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError(
        "Cannot locate this DeepGPR repository. Start Jupyter from the project root or examples folder."
    )
src_path = (project_root / "src").resolve()
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))

import DeepGPR

loaded_package = Path(DeepGPR.__file__).resolve()
if not loaded_package.is_relative_to(src_path):
    raise RuntimeError(
        f"DeepGPR was already imported from {loaded_package}. "
        "Restart the kernel so the project-local package can be loaded."
    )
print("DeepGPR project package:", loaded_package)

if torch.cuda.is_available():
    device = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
else:
    device = torch.device("cpu")

FDTD_ORDER = 2
PML_WIDTH = 10
dx = 0.05
dt = 5.0e-11
nt = 2000
frequency = 2.0e8

source_location = torch.zeros((50, 1, 3), device=device, dtype=torch.int32)
source_location[:, 0, 0] = torch.arange(10, 210, 4, device=device)
source_location[:, :, 1] = 10

receiver_location = torch.zeros((50, 200, 3), device=device, dtype=torch.int32)
receiver_location[:, :, 0] = torch.arange(10, 210, device=device).repeat(50, 1)
receiver_location[:, :, 1] = 110

source_amplitudes = torch.zeros((1, nt, 1), device=device)
source_amplitudes[0, :, 0] = DeepGPR.wavelet.ricker(
    frequency, nt, dt, 1.0 / frequency
).to(device)

model_path = Path.cwd() / "2DFWImodel.npy"
if not model_path.exists():
    model_path = project_root / "examples" / "2DFWImodel.npy"
model = torch.as_tensor(np.load(model_path), dtype=torch.float32, device=device)
er_true = F.pad(model[None, None], (10, 10, 10, 10), mode="replicate")[0, 0]
se_true = er_true.clone() * 1.0e-3
se_true[105:, :] = torch.flip(se_true[105:, :], dims=[1])

assert torch.isfinite(er_true).all() and torch.isfinite(se_true).all()
print("device:", device, "fdtd_order:", FDTD_ORDER, "model shape:", tuple(er_true.shape))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
im1 = axes[0].imshow(er_true[10:-10, 10:-10].cpu(), cmap="jet", vmin=1, vmax=7)
plt.colorbar(im1, ax=axes[0])
axes[0].set_title("True relative permittivity")
axes[0].set_xlabel("Grid index")
axes[0].set_ylabel("Grid index")
im2 = axes[1].imshow(se_true[10:-10, 10:-10].cpu(), cmap="jet", vmin=0.001, vmax=0.007)
plt.colorbar(im2, ax=axes[1], label="S/m")
axes[1].set_title("True conductivity")
axes[1].set_xlabel("Grid index")
axes[1].set_ylabel("Grid index")
plt.tight_layout()
plt.show()

In [ ]:
# 生成观测数据。正演不需要保存反传波场。
with torch.no_grad():
    r_obs = DeepGPR.compute(
        device=device,
        dx=dx,
        dt=dt,
        source_amplitudes=source_amplitudes,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er_true,
        se=se_true,
        pmlthick=PML_WIDTH,
        fdtd_order=FDTD_ORDER,
    )[-1].detach()

assert torch.isfinite(r_obs).all(), "Observed data contain NaN or Inf."
data_scale = r_obs.abs().amax().clamp_min(torch.finfo(r_obs.dtype).tiny)
print("observed max amplitude:", float(data_scale))

plt.figure(figsize=(6, 5))
im = plt.imshow(r_obs[:, :, 100].cpu(), cmap="seismic", origin="lower", aspect="auto")
plt.colorbar(im, label="Amplitude")
plt.title("Signal received by receiver 100")
plt.xlabel("Time index")
plt.ylabel("Shot index")
plt.tight_layout()
plt.show()

In [ ]:
# skimage 在 CPU 上平滑，再将结果送回目标设备，兼容服务器 CUDA 环境。
er_initial = filters.gaussian(
    er_true.detach().cpu().numpy(), sigma=20, preserve_range=True
)
se_initial = filters.gaussian(
    se_true.detach().cpu().numpy(), sigma=20, preserve_range=True
)
er = torch.as_tensor(er_initial, dtype=torch.float32, device=device).clone().requires_grad_(True)
se = torch.as_tensor(se_initial, dtype=torch.float32, device=device).clone().requires_grad_(True)

phase1_epochs = 50
n_epochs = 201

optimizer_phase1 = torch.optim.Adam([er], lr=0.1, eps=1.0e-6)
optimizer_phase2 = torch.optim.Adam(
    [{"params": [er], "lr": 0.05}, {"params": [se], "lr": 5.0e-5}],
    eps=1.0e-6,
)

# TV 直接作为目标函数的一部分，不再按数据梯度幅值进行动态缩放。
TV_WEIGHT_ER = 1.0e-8
TV_WEIGHT_SE = 1.0e-10
tv_criterion = DeepGPR.TVRegularization(
    weight_ep=TV_WEIGHT_ER,
    weight_sigma=TV_WEIGHT_SE,
).to(device)

# CPML 内的材料保持初始值，只反演物理区域。
update_mask = torch.zeros_like(er, dtype=torch.bool)
update_mask[PML_WIDTH:-PML_WIDTH, PML_WIDTH:-PML_WIDTH] = True
er_fixed = er.detach().clone()
se_fixed = se.detach().clone()

def check_model_finite(epoch):
    if not torch.isfinite(er).all() or not torch.isfinite(se).all():
        raise RuntimeError(f"Model became non-finite after epoch {epoch}.")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(er[10:-10, 10:-10].detach().cpu(), cmap="jet", vmin=1, vmax=7)
axes[0].set_title("Initial relative permittivity")
axes[1].imshow(se[10:-10, 10:-10].detach().cpu(), cmap="jet", vmin=0.001, vmax=0.007)
axes[1].set_title("Initial conductivity")
plt.tight_layout()
plt.show()


In [ ]:
history = []
start_time = time.time()

for epoch in range(n_epochs):
    phase1 = epoch < phase1_epochs
    current_optimizer = optimizer_phase1 if phase1 else optimizer_phase2
    er.requires_grad_(True)
    se.requires_grad_(not phase1)
    current_optimizer.zero_grad(set_to_none=True)
    epoch_start = time.time()

    r = DeepGPR.compute(
        device=device,
        dx=dx,
        dt=dt,
        source_amplitudes=source_amplitudes,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=PML_WIDTH,
        model_gradient_sampling_interval=5,
        fdtd_order=FDTD_ORDER,
    )[-1]

    # 数据项与 TV 项组成同一个明确目标函数，只执行一次反传。
    residual = (r - r_obs) / data_scale
    loss_data = F.smooth_l1_loss(residual, torch.zeros_like(residual), beta=0.05)
    loss_tv = tv_criterion(er, se if se.requires_grad else None)
    loss_total = loss_data + loss_tv
    loss_total.backward()

    if er.grad is None or (se.requires_grad and se.grad is None):
        raise RuntimeError(f"Model gradient is missing at epoch {epoch}.")

    # 只屏蔽 CPML 区域；物理区域的原始梯度不做任何尺度处理。
    with torch.no_grad():
        er.grad.masked_fill_(~update_mask, 0.0)
        if se.requires_grad:
            se.grad.masked_fill_(~update_mask, 0.0)

    if not torch.isfinite(er.grad).all() or (se.requires_grad and not torch.isfinite(se.grad).all()):
        raise RuntimeError(f"Gradient became non-finite at epoch {epoch}.")

    raw_max_er = float(er.grad.detach().abs().amax())
    raw_max_se = float(se.grad.detach().abs().amax()) if se.requires_grad else 0.0
    current_optimizer.step()

    with torch.no_grad():
        er.clamp_(1.0, 12.0)
        se.clamp_(0.0, 0.02)
        er[~update_mask] = er_fixed[~update_mask]
        se[~update_mask] = se_fixed[~update_mask]
    check_model_finite(epoch)

    history.append(
        {
            "epoch": epoch,
            "total_loss": float(loss_total),
            "data_loss": float(loss_data),
            "tv_loss": float(loss_tv),
        }
    )
    epoch_time = time.time() - epoch_start
    total_time = time.time() - start_time
    print(
        f"Epoch {epoch:03d} | total={float(loss_total):.6e} "
        f"| data={float(loss_data):.6e} | tv={float(loss_tv):.6e} "
        f"| raw_grad_max(er)={raw_max_er:.3e} | raw_grad_max(se)={raw_max_se:.3e} "
        f"| epoch={epoch_time:.2f}s | elapsed={total_time:.2f}s"
    )

    if epoch % 50 == 0 or epoch == n_epochs - 1:
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        im1 = axes[0].imshow(er[10:-10, 10:-10].detach().cpu(), cmap="jet", vmin=1, vmax=7)
        plt.colorbar(im1, ax=axes[0])
        axes[0].set_title(f"Relative permittivity, epoch {epoch}")
        im2 = axes[1].imshow(se[10:-10, 10:-10].detach().cpu(), cmap="jet", vmin=0.001, vmax=0.007)
        plt.colorbar(im2, ax=axes[1], label="S/m")
        axes[1].set_title(f"Conductivity, epoch {epoch}")
        plt.tight_layout()
        plt.show()
